# 02 — Agentul conversațional (Telegram)

**Echipa:** Tofan Bogdan, Manea Alina-Alexandra, Burcă Alina  
**Materia:** NLP

Acest notebook reproduce flow-ul `Conversational agent` din N8N.

## Ce face

1. Primește un mesaj **text sau voce** pe bot-ul Telegram.
2. Dacă e voce: descarcă fișierul și îl transcrie cu **Whisper** (OpenAI).
3. Trimite textul către agentul **Grok** care:
   - are **memorie de conversație** în MongoDB (`Chat_history`)
   - interoghează **Pinecone** (`fin-news-documents`) pentru context din știrile istorice
4. Răspunde în **aceeași modalitate** (text dacă input-ul a fost text, audio TTS dacă a fost voce).

Notebook-ul are două părți:
- **Test sandbox** — invocă agentul direct, fără Telegram, pentru iterație rapidă.
- **Live polling** — pornește bot-ul real (rulează până oprești kernel-ul).

## 1. Setup

In [2]:
import shutil, os
shutil.rmtree('/content/financial-news-agent', ignore_errors=True)

In [3]:
%cd /content
!git clone -b Fin-news-agent-dev https://github.com/BogdanT54/financial-news-agent.git
%cd /content/financial-news-agent
!pip install -q -r requirements.txt

/content
Cloning into 'financial-news-agent'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 66 (delta 24), reused 61 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 37.80 KiB | 1.64 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/financial-news-agent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 35.5 MB/

In [4]:
import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.insert(0, os.path.abspath(".."))

from src.config import get_settings
settings = get_settings()
print("Pinecone index:", settings.PINECONE_INDEX)
print("Main model    :", settings.MODEL_MAIN_AGENT)
print("DRY_RUN       :", settings.DRY_RUN)

Pinecone index: fin-news-documents
Main model    : x-ai/grok-4.3
DRY_RUN       : True


## 2. Construiește agentul conversațional

Agent = LLM (Grok via OpenRouter) + tool Pinecone (retrieve top 50 articole istorice).

In [5]:
from src.vectorstore import as_retriever_tool
from src.agents import build_conversational_agent, run_conversational_agent

retriever_tool = as_retriever_tool(top_k=50)
executor = build_conversational_agent(retriever_tool)
print("Agent gata.")

Agent gata.


## 3. Test rapid cu input text

Întreabă orice despre piețe, crypto, macro etc. — agentul va interoga Pinecone și va răspunde.

In [11]:
raspuns = run_conversational_agent(
    executor,
    user_message="Care este sentimentul recent pe Bitcoin? Dă-mi 5+ referințe din baza de date.",
)
print(raspuns)

**Sentiment recent pe Bitcoin: predominant negativ / precaut** (bazat pe știri din 25 mai 2026).

Motive principale:
- Semnale bearish (model de trend inversat, ieșiri ETF de ~2,7 mld USD).
- Presiune din influxuri Binance și cerere spot la minim 2025.
- Lichidări futures mari și consolidare sub rezistențe.

**Referințe din baza de date (Pinecone):**
- 10x Research Flags Bearish Bitcoin Signal... (negativ, 0.95)
- Binance Bitcoin inflows are flashing a warning signal (negativ, 0.93)
- Bitcoin Apparent Demand Hits 2025 Low... (negativ, 0.76)
- Bitcoin Rebounds Amid Potential US-Iran Deal... (pozitiv, 0.74)
- Bitcoin Price Climbs Into Resistance As Bears Defend Critical Levels (neutru, 0.82)
- Altcoin Season Index Holds At 33: Bitcoin Still Dominates Market (neutru, 0.90)
- Crypto Futures Liquidations Surpass $127M... (negativ, 0.67)

Aceasta este analiză de știri, nu sfat financiar.


## 4. Test cu input audio (upload .ogg / .mp3)

Încarcă un fișier audio și agentul îl va transcrie cu Whisper, va răspunde și va genera audio TTS.

In [7]:
from src.tts import transcribe_audio, generate_audio_opus
from IPython.display import Audio, display

# Înlocuiește cu path-ul către fișierul tău audio
AUDIO_PATH = "sample.ogg"

if os.path.exists(AUDIO_PATH):
    with open(AUDIO_PATH, "rb") as f:
        audio_in = f.read()
    transcript = transcribe_audio(audio_in, filename=os.path.basename(AUDIO_PATH))
    print("Transcript :", transcript)

    raspuns = run_conversational_agent(executor, user_message=transcript)
    print("\nRăspuns text:\n", raspuns)

    audio_out = generate_audio_opus(raspuns)
    display(Audio(data=audio_out, autoplay=False))
else:
    print(f"Nu există {AUDIO_PATH}. Încarcă un fișier audio și re-rulează celula.")

Nu există sample.ogg. Încarcă un fișier audio și re-rulează celula.


## 5. Live bot polling (opțional)

Celula de mai jos pornește bot-ul Telegram în polling. Va rula până oprești kernel-ul.

**Atenție:** pe Colab este nevoie de `nest_asyncio` pentru că loop-ul async se ciocnește cu cel al notebook-ului.

In [ ]:
# !pip install -q nest_asyncio

# import nest_asyncio; nest_asyncio.apply()
# from src.telegram_io import run_bot_polling, download_voice
# from telegram import Update
# from telegram.ext import ContextTypes
# from src.tts import transcribe_audio, generate_audio_opus
#
# async def handler(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
#     msg = update.message
#     if msg is None:
#         return
#     await context.bot.send_message(chat_id=msg.chat.id, text="Processing...",
#                                    reply_to_message_id=msg.message_id)
#     if msg.voice:
#         audio_bytes = await download_voice(msg.voice.file_id)
#         text_in = transcribe_audio(audio_bytes, filename="voice.ogg")
#         reply = run_conversational_agent(executor, user_message=text_in)
#         audio_out = generate_audio_opus(reply)
#         await context.bot.send_audio(chat_id=msg.chat.id, audio=audio_out,
#                                      filename="reply.opus",
#                                      reply_to_message_id=msg.message_id)
#     else:
#         reply = run_conversational_agent(executor, user_message=msg.text or "")
#         await context.bot.send_message(chat_id=msg.chat.id, text=reply,
#                                        reply_to_message_id=msg.message_id)
#
# run_bot_polling(handler)